# Full local reproduction — 6 November 2026

This copy enables full training and batch evaluation on a new project clone. Earlier diagnostic cells inspect the delivered baseline; the full-rerun cell reports the fresh clone. The companion XAI_Complete_Experiment.ipynb keeps the lighter default configuration.

# Cross-Institution XAI for Network Traffic Forecasting
## Complete experimental notebook — data, models, training, forecasts and temporal explanations

**Versi 6 November 2026.** Pendamping studi *Cross-Institution Reliability of Temporal Attributions in Network Traffic Forecasting*.

Notebook ini memperlihatkan implementasi inti langsung di dalam sel: eligibility windows, Ridge/DLinear/GRU, training loop, inference, exact group Shapley, occlusion, IG, controlled operators, dan paired bootstrap. Tidak perlu membuka script eksternal untuk memahami algoritmanya. Dataset/checkpoint/hasil tetap berada dalam paket proyek agar ukuran notebook wajar.

**Cara memakai:** ekstrak paket lengkap; pilih kernel Python dengan `requirements-full.txt` dan Jupyter/ipykernel; jalankan dari atas. Mode default melakukan EDA, refit Ridge sebagai pemeriksaan reproduksi, inference dari checkpoint, penghitungan atribusi kasus nyata dan bootstrap ulang. Full neural retraining dan batch evaluation tersedia di bagian akhir dengan flag eksplisit dan menulis ke salinan baru.

**Referensi struktur:** [hananlu/xai — XAI_Network_Traffic.ipynb](https://github.com/hananlu/xai/blob/master/XAI_Network_Traffic.ipynb), diakses 6 November 2026. Susunan EDA → preprocessing → modeling → global/local XAI → agreement/stability/latency → robustness menginspirasi penyajian. Implementasi forecasting di sini memakai kode studi kita; tidak menyalin pipeline klasifikasi, random split, label encoding, atau target klasifikasinya.

## 1. Research question dan batas klaim

**RQ primer:** pada frozen GRU, horizon 1 jam, source-calendar Shapley, apakah ranking alignment berubah ketika model dipakai pada institusi yang tidak masuk fitting?

Tiga objek pengukuran dipisahkan:

| Objek | Pertanyaan | Ukuran |
|---|---|---|
| Forecast | Seberapa dekat prediksi dengan volume berikutnya? | MAE, RMSE, MASE |
| Numerik atribusi | Apakah perhitungan memenuhi game/path dan toleransi yang ditetapkan? | Shapley efficiency; IG completeness + refinement |
| Ranking alignment | Apakah ranking mendahulukan hari dengan respons besar di bawah operator? | Q relatif random dan response oracle |

Q bukan probabilitas salah, bukan causal demand attribution dan bukan universal explanation correctness. Semua target sudah pernah dievaluasi; notebook ini mereproduksi dan memperjelas studi, bukan membuka holdout baru. Analisis agreement, latency, visual EDA dan kasus lokal adalah deskriptif tambahan, tidak mengganti endpoint paper.

## 2. Konfigurasi dan dependensi

Python 3.11 digunakan dalam run asli. Di environment baru instal `requirements-full.txt`, ditambah `jupyterlab` dan `ipykernel`, lalu pilih kernel tersebut. Tidak ada pemasangan paket, mounting Drive, download besar atau compute cloud otomatis.

Di Colab, upload dan ekstrak ZIP proyek, lalu isi `PROJECT_DIR`; kompatibilitas Colab belum diuji dalam run lokal ini. Jangan hanya upload notebook tanpa folder data/checkpoint. Full retraining CPU membutuhkan beberapa menit; default notebook lebih ringan.

In [ ]:
from pathlib import Path
import sys, json, time, copy, hashlib, itertools, shutil, gc
from math import factorial
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.polynomial.legendre import leggauss
from scipy.stats import kendalltau
from sklearn.linear_model import Ridge
import torch
from torch import nn
from IPython.display import display, Markdown

PROJECT_DIR = None  # Contoh: "/content/xai_cross_institution"
candidates = [Path(PROJECT_DIR).expanduser()] if PROJECT_DIR else [
    Path.cwd(), Path.cwd()/"xai_cross_institution", Path.cwd()/"outputs/xai_cross_institution"
]
ROOT = next((p.resolve() for p in candidates if (p/"results/NUMERIC_CLAIMS.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Ekstrak paket lengkap dan isi PROJECT_DIR dengan folder xai_cross_institution.")
SOURCE_ROOT = ROOT
NOTEBOOK_PATH = SOURCE_ROOT / "XAI_Full_Local_Run.ipynb"
SEED = 42
HORIZONS = [1, 6]
YEARS = [2025, 2026]
DAY_NAMES = [f"Day {i+1}" for i in range(7)]  # oldest -> newest
RUN_RIDGE_REPRO_CHECK = True
RUN_FULL_TRAINING = True
RUN_FULL_EVALUATION = True
EXPORT_EXPLORATORY = False
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(4)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 30)
plt.rcParams.update({"figure.dpi":110, "axes.spines.top":False, "axes.spines.right":False})
print("Project:", ROOT.name)
print("Python:",sys.version.split()[0], "Torch:",torch.__version__)

### 2.1 Artefak dan pembekuan protokol

Hash lokal melindungi jejak kode dan protokol yang ada sebelum target dibuka; ini bukan public preregistration. Field `target_outcomes_scored:false` pada manifest freeze adalah status historis. Status saat ini ada di `STUDY_COMPLETE.json`.

In [ ]:
split = pd.read_csv(ROOT/"INSTITUTION_SPLIT.csv")
pairs = pd.read_csv(ROOT/"MATCHED_PAIRS.csv")
claims = json.loads((ROOT/"results/NUMERIC_CLAIMS.json").read_text())
training_record = json.loads((ROOT/"results/TRAINING_COMPLETE.json").read_text())
manifest = json.loads((ROOT/"data/MANIFEST.json").read_text())
hash_audit = []
for fname in ["FULL_PREFIT_HASHES.json", "FULL_PREEVALUATION_HASHES.json"]:
    record = json.loads((ROOT/"results"/fname).read_text())
    for rel, expected in record["files"].items():
        actual = hashlib.sha256((ROOT/rel).read_bytes()).hexdigest()
        hash_audit.append({"manifest":fname,"file":rel,"match":actual==expected})
assert all(x["match"] for x in hash_audit)
print("Frozen code/protocol checks passed:",len(hash_audit))
print("Dataset:",manifest["dataset"], "License:",manifest["license"])
print("Training target institutions fitted:",training_record["target_institutions_fitted"])
display(split.groupby("role").size().rename("institutions").to_frame())
display(pairs)

## 3. Membaca dataset dan memeriksa schema

**CESNET-TimeSeries26**: official institutional hourly `n_bytes`, dikonversi ke decimal GB = bytes/1e9. Sumber [Zenodo](https://doi.org/10.5281/zenodo.21945629), CC BY 4.0, Josef Koumar, Karel Hynek, Kamil Jeřábek, Pavel Šiška (2026). Institusi berasal dari satu ISP/research-network ecosystem.

`complete_hour` berasal dari cakupan enam native 10-minute timestamps; keberadaan nilai institusi tetap diperiksa terpisah. `conservative_artifact_flag` adalah screening global dengan scope institusi yang tidak pasti. Nilai nol teramati dipertahankan; missing bukan nol.

In [ ]:
example_id = int(split.query("role == 'source'").id.iloc[0])
example_df = pd.read_csv(ROOT/f"data/institution_{example_id}.csv.gz")
print("Schema:")
print(example_df.dtypes)
print("Shape:",example_df.shape)
display(example_df.head())

In [ ]:
quality_rows=[]
for item in manifest["files"]:
    path = ROOT/"data"/item["file"]
    assert hashlib.sha256(path.read_bytes()).hexdigest()==item["sha256"]
    d=pd.read_csv(path)
    y=d.n_bytes.to_numpy(float)/1e9
    finite=np.isfinite(y)
    complete=d.complete_hour.to_numpy(bool)
    flagged=d.conservative_artifact_flag.to_numpy(bool)
    quality_rows.append(dict(id=item["id"],role=item["role"],hours=len(d),
        missing=int((~finite).sum()),observed_zero=int(((y==0)&finite).sum()),
        negative=int(((y<0)&finite).sum()),duplicate_time=int(d.time.duplicated().sum()),
        complete_native=int(complete.sum()),
        usable_hours=int((finite&(y>=0)&complete&~flagged).sum()),
        median_gb=float(np.nanmedian(y))))
quality_df=pd.DataFrame(quality_rows)
assert quality_df.duplicate_time.sum()==0
print("All 48 derived file hashes passed.")
display(quality_df.groupby("role").agg(institutions=("id","size"),
    recorded_grid_hours=("hours","sum"),missing=("missing","sum"),
    observed_zero=("observed_zero","sum"),negative=("negative","sum"),
    usable_hours=("usable_hours","sum")))
display(quality_df.sort_values("missing",ascending=False).head(10))

### 3.1 Coverage per institusi

Plot coverage menjelaskan selection pressure dari complete-history rule. Ini data availability, bukan skor model. Tidak membuang institusi setelah melihat error.

In [ ]:
z=quality_df.sort_values(["role","id"]).reset_index(drop=True)
fig,ax=plt.subplots(figsize=(13,4),constrained_layout=True)
colors=z.role.map({"source":"#2365a0","validation":"#0d9488","target":"#c16b24"})
ax.bar(np.arange(len(z)),z.usable_hours/z.hours,color=colors)
ax.set_xticks(np.arange(len(z)),[f"{r.id}" for r in z.itertuples()],rotation=90)
ax.set_ylim(0,1.05);ax.set_ylabel("Fraction of usable hours")
ax.set_xlabel("Institution ID; blue=source, green=validation, orange=target")
ax.set_title("Coverage before requiring a complete 168-hour history")
plt.show()

### 3.2 Time-series dan pola kalender

Pilih satu institusi per role berdasarkan urutan membership, bukan performa. Garis log1p hanya untuk visualisasi; gap tidak diisi. Heatmap kalender memakai **source 2024 saja**, bukan target atau tahun evaluasi.

In [ ]:
fig,axes=plt.subplots(3,1,figsize=(13,7),sharex=True,constrained_layout=True)
for ax,role in zip(axes,["source","validation","target"]):
    ident=int(split.query("role==@role").id.iloc[0])
    d=pd.read_csv(ROOT/f"data/institution_{ident}.csv.gz")
    t=pd.to_datetime(d.time,utc=True)
    ax.plot(t,np.log1p(d.n_bytes/1e9),lw=.6)
    ax.set_ylabel("log1p(GB)");ax.set_title(f"{role}: institution {ident}")
axes[-1].set_xlabel("UTC timestamp")
plt.show()
profile=np.load(ROOT/"models/source_profile.npy")
fig,ax=plt.subplots(figsize=(11,3),constrained_layout=True)
im=ax.imshow(np.log1p(profile.reshape(7,24)),aspect="auto",cmap="viridis")
ax.set_yticks(range(7),["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])
ax.set_xticks(range(0,24,3));ax.set_xlabel("UTC hour");ax.set_title("Source-only 2024 hour-of-week median profile")
fig.colorbar(im,ax=ax,label="log1p(GB)");plt.show()

## 4. Sampling, institution split dan pencegahan leakage

32 source / 8 validation / 8 target, di luar 12 institusi development terdahulu. Membership menggunakan saved 2024 coverage/volume metadata, empat volume strata dan deterministic hashing. Pair matching memakai 2024 median volume dan coefficient of variation; matching tidak menghilangkan confounding.

Training target timestamps: 2024. Validation: institusi validation, Jan–Mar 2025. Evaluation: eligible endpoints Apr 8–Jul 1 pada 2025 dan 2026, origin UTC hour divisible by 6. Tidak ada random row split lintas waktu. Metadata target 2024 dipakai untuk selection/matching/metric scaling, **bukan** gradient fitting; ini bukan berarti target sama sekali tidak pernah dipakai untuk metadata.

In [ ]:
roles={role:set(split.query("role==@role").id) for role in split.role.unique()}
assert split.id.is_unique
assert not (roles["source"] & roles["validation"])
assert not (roles["source"] & roles["target"])
assert not (roles["validation"] & roles["target"])
assert set(pairs.source)<=roles["source"] and set(pairs.target)==roles["target"]
fig,ax=plt.subplots(figsize=(7,3),constrained_layout=True)
for role,color in [("source","#2365a0"),("validation","#0d9488"),("target","#c16b24")]:
    vals=quality_df.query("role==@role").median_gb
    ax.scatter(np.full(len(vals),role),np.log1p(vals),label=role,color=color,alpha=.8)
ax.set_ylabel("log1p(median observed GB)");ax.set_title("Volume heterogeneity; plot is descriptive")
plt.show()

## 5. Preprocessing dan pembentukan supervised windows — kode inti

Setiap example memakai 168 jam sampai origin o. Label adalah y[o+1] dan y[o+6], bukan seluruh lintasan enam jam. Seluruh history dan kedua endpoint harus usable. Interval periode ditentukan oleh target timestamps; history boleh berada sebelum awal interval.

In [ ]:
def windows(path):
    d = pd.read_csv(path)
    t = pd.DatetimeIndex(pd.to_datetime(d.time, utc=True))
    y = d.n_bytes.to_numpy(float) / 1000000000.0
    valid = d.complete_hour.to_numpy(bool) & ~d.conservative_artifact_flag.to_numpy(bool) & np.isfinite(y) & (y >= 0)
    o = np.arange(167, len(y) - 6)
    ok = np.ones(len(o), bool)
    for j in range(168):
        ok &= valid[o - j]
    for h in [1, 6]:
        ok &= valid[o + h]
    o = o[ok]
    X = np.stack([y[a - 167:a + 1] for a in o])
    Y = np.stack([y[o + h] for h in [1, 6]], 1)
    return dict(X=X, Y=Y, o=o, t=t, y=y, valid=valid)

def period(v, start, end, stride=None):
    t, o = (v['t'], v['o'])
    q = (t[o + 1] >= pd.Timestamp(start, tz='UTC')) & (t[o + 6] < pd.Timestamp(end, tz='UTC'))
    if stride:
        q &= t[o].hour % stride == 0
    return q

In [ ]:
selection=pd.read_csv(ROOT/"results/explanation_selection.csv")
# Case dipilih dengan metadata tetap, tanpa mencari error/quality tertentu.
case=selection.query("pair==3 and year==2025 and cohort=='target'").sort_values("origin").iloc[0]
case_id=int(case.id)
case_v=windows(ROOT/f"data/institution_{case_id}.csv.gz")
case_o=int(case_v["t"].get_loc(pd.Timestamp(case.origin)))
case_ix=int(np.flatnonzero(case_v["o"]==case_o)[0])
x_case=case_v["X"][case_ix]
y_case=case_v["Y"][case_ix]
assert x_case.shape==(168,) and y_case.shape==(2,)
assert np.all(np.isfinite(x_case)) and np.all(x_case>=0)
assert np.all(case_v["valid"][case_o-167:case_o+1])
assert all(case_v["t"][case_o+h]>case_v["t"][case_o] for h in HORIZONS)
print("Case metadata:",case[["pair","id","cohort","year","origin"]].to_dict())
print("X shape",x_case.shape,"Y shape",y_case.shape)
print("Observed endpoint GB:",dict(zip(HORIZONS,y_case)))

In [ ]:
fig,ax=plt.subplots(figsize=(12,3.5),constrained_layout=True)
ax.plot(np.arange(-167,1),x_case,label="168h observed history",lw=1)
ax.scatter(HORIZONS,y_case,c="#c16b24",label="Observed future endpoints",zorder=5)
ax.axvline(0,c="gray",ls="--");ax.set_xlabel("Hours relative to origin")
ax.set_ylabel("GB");ax.set_title(f"Institution {case_id}, {case.origin}")
ax.legend();plt.show()

### 5.1 Memuat data fitting dan normalization source-only

Array log1p float32 sama dengan training asli; Ridge difit dalam float64 dari array tersebut. Validation loss dirata-rata per institusi, bukan didominasi institusi yang punya banyak windows. Tidak memuat target ke array fitting. Cell ini merekonstruksi input fitting, tanpa neural retraining.

In [ ]:
source_arrays=[]; validation_arrays=[]; fit_count_rows=[]
for row in split.itertuples():
    if row.role=="target":
        continue
    v=windows(ROOT/f"data/institution_{row.id}.csv.gz")
    mask=period(v,"2024-01-01","2025-01-01") if row.role=="source" else period(v,"2025-01-01","2025-04-01")
    xx=np.log1p(v["X"][mask]).astype("float32")
    yy=np.log1p(v["Y"][mask]).astype("float32")
    assert len(xx)>0
    (source_arrays if row.role=="source" else validation_arrays).append((int(row.id),xx,yy))
    fit_count_rows.append(dict(id=int(row.id),role=row.role,windows=len(xx)))
    del v
X_source=np.concatenate([a for _,a,_ in source_arrays])
Y_source=np.concatenate([b for _,_,b in source_arrays])
source_mean=float(X_source.mean());source_std=float(X_source.std())
assert len(X_source)==training_record["source_windows"]
assert abs(source_mean-training_record["normalization_mean"])<1e-8
assert abs(source_std-training_record["normalization_std"])<1e-8
assert {i for i,_,_ in source_arrays}<=roles["source"]
assert {i for i,_,_ in validation_arrays}<=roles["validation"]
print("Source arrays:",X_source.shape,Y_source.shape)
print("Source-only mean/std:",source_mean,source_std)
display(pd.DataFrame(fit_count_rows).groupby("role").agg(institutions=("id","size"),windows=("windows","sum")))

## 6. Ridge: fit langsung dan pemeriksaan checkpoint

Hyperparameter yang sudah dipilih adalah alpha=0.1 per horizon. Sel berikut **benar-benar memfit ulang** Ridge pada source 2024 dengan setting yang dibekukan, lalu membandingkan koefisien dengan checkpoint. Tidak mencari hyperparameter baru dari target. Set `RUN_RIDGE_REPRO_CHECK=False` untuk melewatinya.

In [ ]:
ridge_checkpoint=np.load(ROOT/"models/ridge_0.npz")
if RUN_RIDGE_REPRO_CHECK:
    ridge_refits=[]
    t0=time.perf_counter()
    for k,h in enumerate(HORIZONS):
        model=Ridge(alpha=.1,solver="cholesky").fit(X_source.astype("float64"),Y_source[:,k].astype("float64"))
        ridge_refits.append(model)
        diff=np.max(np.abs(model.coef_-ridge_checkpoint["coef"][k]))
        assert np.allclose(model.coef_,ridge_checkpoint["coef"][k],rtol=1e-7,atol=1e-9)
        val_loss=np.mean([np.mean(np.abs(model.predict(a)-b[:,k])) for _,a,b in validation_arrays])
        print(f"h={h}: alpha=.1, validation macro log-MAE={val_loss:.6f}, max coef difference={diff:.3g}")
    print(f"Ridge refit check: {time.perf_counter()-t0:.2f}s")
else:
    print("Ridge refit check disabled; using delivered checkpoint.")

## 7. DLinear dan GRU — arsitektur dan fungsi yang diatribusikan

DLinear: moving average 25 jam dengan endpoint replication, trend/seasonal heads 168→2. GRU: satu layer, 32 hidden units, correction terhadap last log input; head awal nol. Output GB = expm1(max(0,predicted_log)). Preprocessing dan clipping termasuk dalam fungsi atribusi; IG tidak berhenti di latent/log output.

In [ ]:
class Forecaster(nn.Module):

    def __init__(self, kind, mean, std):
        super().__init__()
        self.kind = kind
        self.register_buffer('mean', torch.tensor(float(mean)))
        self.register_buffer('std', torch.tensor(float(std)))
        if kind == 'gru':
            self.gru = nn.GRU(1, 32, batch_first=True)
            self.head = nn.Linear(32, 2)
            nn.init.zeros_(self.head.weight)
            nn.init.zeros_(self.head.bias)
        else:
            self.trend = nn.Linear(168, 2)
            self.season = nn.Linear(168, 2)
            for m in [self.trend, self.season]:
                nn.init.constant_(m.weight, 1 / 168)
                nn.init.zeros_(m.bias)

    def log_prediction(self, x):
        z = (x - self.mean) / self.std
        if self.kind == 'gru':
            _, h = self.gru(z.unsqueeze(-1))
            return x[:, -1, None] + self.head(h[-1]) * self.std
        trend = torch.nn.functional.avg_pool1d(torch.nn.functional.pad(z[:, None, :], (12, 12), mode='replicate'), 25, 1)[:, 0, :]
        return (self.trend(trend) + self.season(z - trend)) * self.std + self.mean

    def forward(self, gb):
        return torch.expm1(torch.clamp_min(self.log_prediction(torch.log1p(gb)), 0))

In [ ]:
architecture_rows=[]
for kind in ["dlinear","gru"]:
    checkpoint=torch.load(ROOT/f"models/{kind}_42.pt",map_location="cpu",weights_only=True)
    model=Forecaster(kind,checkpoint["mean"],checkpoint["std"])
    model.load_state_dict(checkpoint["state"]);model.double().eval()
    with torch.no_grad():pred=model(torch.tensor(x_case[None,:],dtype=torch.float64)).numpy()
    architecture_rows.append(dict(model=kind,trainable_parameters=sum(p.numel() for p in model.parameters()),
                                  input_hours=168,output_endpoints=2,prediction_h1=float(pred[0,0]),prediction_h6=float(pred[0,1])))
    print(kind,model)
display(pd.DataFrame(architecture_rows))

### 7.1 Training loop lengkap — implementasi inline

Kode di bawah memuat fitting Ridge grid, balanced institution sampling, Adam, log-MSE, gradient clipping, validation per institusi, patience, pemilihan learning rate memakai seed 42, dan training seeds 123/2026. Definisi tidak otomatis melatih model. Bagian akhir mengaktifkannya hanya pada salinan baru.

Konfigurasi fixed: batch 256 (32×1024 examples/epoch dibagi 128 batches), maksimum 30 epochs, patience 5; LR {0.001,0.0003}. Pemilihan LR tidak diulang memakai target. `source_sha256` pada inline rerun menunjuk hash notebook yang dijalankan, bukan mengklaim file script historis dieksekusi.

In [ ]:
def save_model(kind, seed, model, meta):
    p = ROOT / f'models/{kind}_{seed}.pt'
    torch.save({'kind': kind, 'state': model.state_dict(), 'mean': float(model.mean), 'std': float(model.std), 'metadata': meta}, p)
    return str(p.name)

In [ ]:
def train_all_models_inline():
    start = time.time()
    (ROOT / 'models').mkdir(exist_ok=True)
    split = pd.read_csv(ROOT / 'INSTITUTION_SPLIT.csv')
    source = []
    validation = []
    profile_y = []
    profile_h = []
    counts = []
    for row in split.itertuples():
        if row.role == 'target':
            continue
        v = windows(ROOT / f'data/institution_{row.id}.csv.gz')
        q = period(v, '2024-01-01', '2025-01-01') if row.role == 'source' else period(v, '2025-01-01', '2025-04-01')
        x = np.log1p(v['X'][q]).astype('float32')
        y = np.log1p(v['Y'][q]).astype('float32')
        assert len(x) > 0
        counts.append(dict(id=row.id, role=row.role, n=len(x)))
        if row.role == 'source':
            source.append((row.id, x, y))
            t = v['t']
            ok = (t >= pd.Timestamp('2024-01-01', tz='UTC')) & (t < pd.Timestamp('2025-01-01', tz='UTC')) & v['valid']
            profile_y.extend(v['y'][ok])
            profile_h.extend((t.day_of_week.to_numpy() * 24 + t.hour.to_numpy())[ok])
        else:
            validation.append((row.id, torch.from_numpy(x), torch.from_numpy(y)))
    X = np.concatenate([a[1] for a in source])
    Y = np.concatenate([a[2] for a in source])
    mean = float(X.mean())
    std = float(X.std())
    profile_y = np.array(profile_y)
    profile_h = np.array(profile_h)
    profile = np.array([np.median(profile_y[profile_h == j]) for j in range(168)])
    np.save(ROOT / 'models/source_profile.npy', profile)
    pd.DataFrame(counts).to_csv(ROOT / 'results/full_fit_counts.csv', index=False)
    choices = []
    ridge_coef = []
    ridge_intercept = []
    for k, h in enumerate([1, 6]):
        candidates = []
        for alpha in [0.1, 1, 10, 100]:
            m = Ridge(alpha=alpha, solver='cholesky').fit(X.astype('float64'), Y[:, k].astype('float64'))
            loss = float(np.mean([np.mean(abs(m.predict(a.numpy()) - b.numpy()[:, k])) for _, a, b in validation]))
            candidates.append((loss, alpha, m))
            choices.append(dict(kind='ridge_trial', seed=0, horizon=h, alpha=alpha, validation_log_mae=loss))
        loss, alpha, m = min(candidates, key=lambda z: (z[0], z[1]))
        ridge_coef.append(m.coef_)
        ridge_intercept.append(m.intercept_)
        choices.append(dict(kind='ridge_selected', seed=0, horizon=h, alpha=alpha, validation_log_mae=loss))
    np.savez_compressed(ROOT / 'models/ridge_0.npz', coef=np.stack(ridge_coef), intercept=np.array(ridge_intercept))
    history = []

    def fit(kind, seed, lr):
        torch.manual_seed(seed)
        rng = np.random.default_rng(seed)
        model = Forecaster(kind, mean, std)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        best = np.inf
        state = None
        stale = 0
        best_epoch = 0
        t0 = time.time()
        for epoch in range(1, 31):
            model.train()
            slices = []
            for _, a, b in source:
                ix = rng.choice(len(a), 1024, replace=True)
                slices.append((a[ix], b[ix]))
            ax = np.concatenate([a for a, b in slices])
            by = np.concatenate([b for a, b in slices])
            order = rng.permutation(len(ax))
            losses = []
            for ix in np.array_split(order, 128):
                opt.zero_grad()
                pred = model.log_prediction(torch.from_numpy(ax[ix]))
                loss = ((pred - torch.from_numpy(by[ix])) ** 2).mean()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                losses.append(float(loss.detach()))
            model.eval()
            with torch.no_grad():
                score = float(np.mean([float(abs(model.log_prediction(a) - b).mean()) for _, a, b in validation]))
            history.append(dict(kind=kind, seed=seed, lr=lr, epoch=epoch, train_mse=np.mean(losses), validation_macro_log_mae=score))
            if score < best - 1e-12:
                best = score
                state = copy.deepcopy(model.state_dict())
                stale = 0
                best_epoch = epoch
            else:
                stale += 1
            print(kind, seed, lr, 'epoch', epoch, 'val', round(score, 6), 'elapsed', round(time.time() - t0, 1), flush=True)
            pd.DataFrame(history).to_csv(ROOT / 'results/training_history.csv', index=False)
            if stale >= 5:
                break
        model.load_state_dict(state)
        return (model, dict(kind=kind, seed=seed, learning_rate=lr, best_epoch=best_epoch, validation_log_mae=best, runtime_seconds=time.time() - t0))
    for kind in ['dlinear', 'gru']:
        candidates = []
        for lr in [0.001, 0.0003]:
            model, meta = fit(kind, 42, lr)
            candidates.append((meta['validation_log_mae'], lr, model, meta))
            choices.append(meta | {'phase': 'lr_trial'})
        _, lr, model, meta = min(candidates, key=lambda v: (v[0], v[1]))
        save_model(kind, 42, model, meta)
        choices.append(meta | {'phase': 'selected'})
        for seed in [123, 2026]:
            model, meta = fit(kind, seed, lr)
            save_model(kind, seed, model, meta)
            choices.append(meta | {'phase': 'selected'})
    (ROOT / 'results/MODEL_SELECTION.json').write_text(json.dumps(choices, indent=2))
    facts = dict(status='training_complete', runtime_seconds=time.time() - start, source_institutions=len(source), validation_institutions=len(validation), target_institutions_fitted=0, source_windows=len(X), normalization_mean=mean, normalization_std=std, models=[dict(file=p.name, sha256=hashlib.sha256(p.read_bytes()).hexdigest()) for p in sorted((ROOT / 'models').glob('*'))], source_sha256=hashlib.sha256(Path(NOTEBOOK_PATH).read_bytes()).hexdigest())
    (ROOT / 'results/TRAINING_COMPLETE.json').write_text(json.dumps(facts, indent=2))
    print(json.dumps(facts, indent=2))

### 7.2 Learning curves dan pilihan model tersimpan

Kurva seed/LR adalah jejak training asli, bukan kurva yang dipilih setelah target dibuka. Epoch terakhir bukan otomatis checkpoint terbaik; lihat metadata selected epoch.

In [ ]:
history=pd.read_csv(ROOT/"results/training_history.csv")
model_selection=pd.DataFrame(json.loads((ROOT/"results/MODEL_SELECTION.json").read_text()))
display(model_selection[model_selection.get("phase",pd.Series(index=model_selection.index,dtype=str)).eq("selected") |
                        model_selection.kind.eq("ridge_selected")])
fig,axes=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
for ax,kind in zip(axes,["dlinear","gru"]):
    for (seed,lr),g in history.query("kind==@kind").groupby(["seed","lr"]):
        ax.plot(g.epoch,g.validation_macro_log_mae,label=f"seed={seed}, lr={lr:g}")
    ax.set_title(kind);ax.set_xlabel("Epoch");ax.set_ylabel("Validation macro log-MAE");ax.legend(fontsize=8)
plt.show()

## 8. Inference wrapper, naive baselines dan calibration — kode inti

Wrapper mengembalikan dua endpoint sekaligus dan memuat checkpoint tanpa target-specific normalization. Calibration adalah offset median log-residual dari eligible Apr 1–8 dengan minimal 24 examples; bukan neural fine-tuning dan bukan arm zero-shot.

In [ ]:
class RidgeModule(nn.Module):

    def __init__(self, coef, intercept):
        super().__init__()
        self.register_buffer('coef', torch.tensor(coef, dtype=torch.float64))
        self.register_buffer('intercept', torch.tensor(intercept, dtype=torch.float64))

    def log_prediction(self, x):
        return x @ self.coef.T + self.intercept

    def forward(self, x):
        return torch.expm1(torch.clamp_min(self.log_prediction(torch.log1p(x)), 0))

class Predictor:

    def __init__(self, kind, seed):
        self.kind = kind
        self.seed = seed
        if kind == 'ridge':
            m = np.load(ROOT / 'models/ridge_0.npz')
            self.net = RidgeModule(m['coef'], m['intercept'])
        else:
            d = torch.load(ROOT / f'models/{kind}_{seed}.pt', weights_only=True, map_location='cpu')
            self.net = Forecaster(kind, d['mean'], d['std'])
            self.net.load_state_dict(d['state'])
            self.net.double()
        self.net.eval()

    def __call__(self, x, bias=None, rawlog=False):
        x = np.asarray(x, dtype=float)
        flat = x.reshape(-1, 168)
        chunks = []
        with torch.no_grad():
            for a in range(0, len(flat), 512):
                t = torch.tensor(flat[a:a + 512], dtype=torch.float64)
                z = self.net.log_prediction(torch.log1p(t))
                if bias is not None:
                    z = z + torch.tensor(bias, dtype=torch.float64)
                y = z if rawlog else torch.expm1(torch.clamp_min(z, 0))
                chunks.append(y.numpy())
        return np.concatenate(chunks).reshape(x.shape[:-1] + (2,))

In [ ]:
def denominator(v):
    t, y, ok = (v['t'], v['y'], v['valid'])
    ix = np.arange(168, len(y))
    q = (t[ix] >= pd.Timestamp('2024-01-01', tz='UTC')) & (t[ix] < pd.Timestamp('2025-01-01', tz='UTC')) & ok[ix] & ok[ix - 168]
    d = float(np.mean(abs(y[ix[q]] - y[ix[q] - 168])))
    return (max(d, 1e-09), bool(d < 1e-09))

def naive(v, q, kind):
    o = v['o'][q]
    y = v['y']
    if kind == 'persistence':
        return np.repeat(y[o, None], 2, axis=1)
    lag = 24 if kind == 'daily' else 168
    return np.stack([y[o + h - lag] for h in [1, 6]], axis=1)

def calibration(v, year, model=None, kind=None):
    q = period(v, f'{year}-04-01', f'{year}-04-08')
    n = int(q.sum())
    if n < 24:
        return (np.zeros(2), n)
    predlog = model(v['X'][q], rawlog=True) if model is not None else np.log1p(naive(v, q, kind))
    return (np.median(np.log1p(v['Y'][q]) - predlog, axis=0), n)

In [ ]:
predictors={kind:Predictor(kind,0 if kind=="ridge" else 42) for kind in ["ridge","dlinear","gru"]}
case_predictions=[]
for kind,model in predictors.items():
    pred=model(x_case)
    assert pred.shape==(2,) and np.isfinite(pred).all() and (pred>=0).all()
    for k,h in enumerate(HORIZONS):case_predictions.append(dict(model=kind,horizon=h,observed_gb=y_case[k],predicted_gb=pred[k],absolute_error_gb=abs(pred[k]-y_case[k])))
display(pd.DataFrame(case_predictions))

## 9. Forecast metrics: per-cell, institution dan macro

MASE denominator adalah observed eligible 2024 weekly absolute difference per institusi, floored 1e-9 GB. Raw GB errors tidak menghilangkan perbedaan volume. Angka headline adalah mean available institution-year cells setelah seed averaging (15 target cells), bukan balanced panel. Semua per-origin predictions disediakan agar metrics dapat dihitung ulang.

In [ ]:
metrics=pd.read_csv(ROOT/"results/forecast_metrics.csv")
metric_cols=["mae_gb","rmse_gb","mase"]
metric_cells=metrics.groupby(["model","id","role","year","arm","horizon"])[metric_cols].mean().reset_index()
macro=metric_cells.groupby(["model","role","arm","horizon"])[metric_cols].mean().reset_index()
target_macro=macro.query("role=='target' and arm=='frozen'")
display(target_macro.round(6))
print("Target institution-year cells:",len(metric_cells.query("model=='ridge' and role=='target' and arm=='frozen' and horizon==1")))

In [ ]:
names=["persistence","daily","weekly","ridge","dlinear","gru"]
fig,axes=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
for ax,h in zip(axes,HORIZONS):
    z=target_macro.query("horizon==@h").set_index("model").reindex(names)
    ax.bar(names,z.mase,color=["#94a3b8"]*3+["#2365a0","#0d9488","#c16b24"])
    ax.set_title(f"Target frozen: {h}h endpoint");ax.set_ylabel("Macro MASE")
    ax.tick_params(axis="x",rotation=30);ax.grid(axis="y",alpha=.2)
plt.show()

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
for ax,h in zip(axes,HORIZONS):
    z=metric_cells.query("role=='target' and arm=='frozen' and horizon==@h")
    vals=[z.query("model==@kind").mase.values for kind in names]
    ax.boxplot(vals,tick_labels=names,showfliers=True)
    ax.tick_params(axis="x",rotation=30);ax.set_ylabel("Institution-year MASE")
    ax.set_title(f"Heterogeneity: {h}h endpoint")
plt.show()

### 9.1 Error analysis dan prediksi kronologis

Contoh institusi/case dipilih sebelumnya melalui metadata. Scatter dan residual menggunakan GRU seed 42; bukan ensemble tiga seed dan bukan hasil pemilihan kasus dengan error tertentu.

In [ ]:
gru_predictions=pd.read_csv(ROOT/"predictions/gru_42.csv.gz")
case_forecasts=gru_predictions.query("id==@case_id and year==2025 and arm=='frozen' and horizon==1").copy()
assert len(case_forecasts)>0
case_forecasts["origin"]=pd.to_datetime(case_forecasts.origin,utc=True)
residual=case_forecasts.prediction_gb-case_forecasts.target_gb
fig,axes=plt.subplots(1,3,figsize=(14,3.5),constrained_layout=True)
axes[0].plot(case_forecasts.origin,case_forecasts.target_gb,label="Observed",lw=1)
axes[0].plot(case_forecasts.origin,case_forecasts.prediction_gb,label="Forecast",lw=1,alpha=.8)
axes[0].tick_params(axis="x",rotation=35);axes[0].set_ylabel("GB");axes[0].legend()
axes[1].scatter(np.log1p(case_forecasts.target_gb),np.log1p(case_forecasts.prediction_gb),s=10,alpha=.6)
axes[1].set_xlabel("Observed log1p(GB)");axes[1].set_ylabel("Forecast log1p(GB)")
axes[2].hist(residual,bins=25,color="#2365a0");axes[2].axvline(0,c="gray")
axes[2].set_xlabel("Forecast − observed GB");axes[2].set_ylabel("Count")
fig.suptitle(f"Institution {case_id}, 2025, frozen GRU seed 42, h1")
plt.show()

In [ ]:
# Recompute a full stored institution/year/arm/horizon cell from per-origin predictions.
saved=metrics.query("model=='gru' and seed==42 and id==@case_id and year==2025 and arm=='frozen' and horizon==1").iloc[0]
recomputed={"mae_gb":float(abs(residual).mean()),"rmse_gb":float(np.sqrt(np.mean(residual**2))),
            "mase":float(abs(residual).mean()/saved.mase_denominator_gb)}
for key,value in recomputed.items():assert np.isclose(value,saved[key],rtol=1e-10,atol=1e-12)
print("Per-origin metric recalculation passed:",recomputed)

## 10. Dua reference atribusi

**Source calendar:** profile median pooled source training 2024 menurut hour of week. **History hour median:** median tujuh hari pada jam yang sama, diulang sepanjang history. Keduanya menjelaskan kontras F(x)−F(r) yang berbeda. Mengganti reference bukan otomatis memperbaiki penjelasan.

In [ ]:
history_times=case_v["t"][case_o-167:case_o+1]
how=history_times.day_of_week.to_numpy()*24+history_times.hour.to_numpy()
references={"source_calendar":profile[how],
            "history_hour_median":np.tile(np.median(x_case.reshape(7,24),axis=0),7)}
fig,ax=plt.subplots(figsize=(12,4),constrained_layout=True)
ax.plot(x_case,label="Observed history",c="black",lw=1)
for name,r in references.items():ax.plot(r,label=name,alpha=.8)
ax.set_xlabel("History hour (oldest → newest)");ax.set_ylabel("GB");ax.legend();plt.show()
contrast_rows=[]
for kind,model in predictors.items():
    for name,r in references.items():
        for k,h in enumerate(HORIZONS):contrast_rows.append(dict(model=kind,reference=name,horizon=h,
            prediction_gb=float(model(x_case)[k]),reference_prediction_gb=float(model(r)[k]),contrast_gb=float(model(x_case)[k]-model(r)[k])))
display(pd.DataFrame(contrast_rows))

## 11. Exact seven-group Shapley dan occlusion — kode inti

Setiap grup adalah 24 jam berturut-turut, oldest→newest. Semua 128 retained-group coalitions dievaluasi; ini exact Shapley untuk game/reference itu, bukan conditional Shapley terhadap distribusi traffic. Occlusion memakai absolute single-group prediction response. Ranking Shapley memakai absolute signed sum per hari, sehingga cancellation di dalam hari masih mungkin.

In [ ]:
MASKS=((np.arange(128)[:,None]>>np.arange(7))&1).astype(float)
PERMUTATIONS=np.array(list(itertools.permutations(range(7))),dtype=int)
REMOVED=np.column_stack([np.zeros(5040,dtype=int),np.cumsum(1<<PERMUTATIONS,axis=1)])
SHAPLEY_WEIGHTS=[]
for j in range(7):
    ix=np.where(MASKS[:,j]==0)[0]
    size=MASKS[ix].sum(1).astype(int)
    weights=np.array([factorial(k)*factorial(6-k)/factorial(7) for k in size])
    SHAPLEY_WEIGHTS.append((ix,weights))
print("Coalitions:",MASKS.shape,"Rankings:",PERMUTATIONS.shape)

In [ ]:
def games(model, x, r, bias=None):
    coal = r[None, :] + np.repeat(MASKS, 24, axis=1) * (x - r)[None, :]
    v = model(coal, bias)
    phi = np.zeros((2, 7))
    for j, (ix, w) in enumerate(SHAPLEY_WEIGHTS):
        phi[:, j] = np.sum((v[ix + (1 << j)] - v[ix]) * w[:, None], axis=0)
    err = abs(phi.sum(1) - (v[-1] - v[0]))
    assert (err <= 1e-08 + 1e-06 * abs(v[-1] - v[0])).all()
    occ = abs(v[-1] - v[127 - (1 << np.arange(7))])
    return (phi, occ.T, v, err)

def order(scores, key):
    tie = np.array([int(hashlib.sha256(f'{key}:{j}'.encode()).hexdigest()[:12], 16) for j in range(7)])
    return np.lexsort((tie, -np.asarray(scores)))

In [ ]:
local_attributions={}; local_rows=[]
for kind,model in predictors.items():
    for ref,r in references.items():
        phi,occ,coalition_values,efficiency_error=games(model,x_case,r)
        local_attributions[(kind,ref)]={"shapley":phi,"occlusion":occ,"game":coalition_values}
        for k,h in enumerate(HORIZONS):
            local_rows.append(dict(model=kind,reference=ref,horizon=h,sum_phi_gb=float(phi[k].sum()),
                                  contrast_gb=float(model(x_case)[k]-model(r)[k]),efficiency_error_gb=float(efficiency_error[k])))
local_numerics=pd.DataFrame(local_rows)
display(local_numerics)
assert local_numerics.efficiency_error_gb.max()<1e-9

### 11.1 Penjelasan lokal: nilai bertanda dan ranking

Panel pertama menjelaskan arah kredit relatif reference. Panel kedua menampilkan magnitude occlusion, yang tidak memiliki semantik kredit bertanda yang sama. Tidak membandingkan tinggi kedua panel sebagai estimator kuantitas identik.

In [ ]:
fig,axes=plt.subplots(2,3,figsize=(14,6),constrained_layout=True)
for col,kind in enumerate(["ridge","dlinear","gru"]):
    phi=local_attributions[(kind,"source_calendar")]["shapley"][0]
    occ=local_attributions[(kind,"source_calendar")]["occlusion"][0]
    axes[0,col].bar(DAY_NAMES,phi,color=np.where(phi>=0,"#2365a0","#c16b24"))
    axes[0,col].axhline(0,c="gray",lw=.7);axes[0,col].set_title(f"{kind}: signed Shapley, h1")
    axes[1,col].bar(DAY_NAMES,occ,color="#0d9488");axes[1,col].set_title("Absolute single-day occlusion response")
    for row in [0,1]:axes[row,col].tick_params(axis="x",rotation=40);axes[row,col].set_ylabel("GB")
plt.show()

### 11.2 Independent permutation identity check

Untuk satu game nonlinear GRU, hitung rata-rata marginal contribution di seluruh 7! permutations secara terpisah dari subset-weight formula. Pemeriksaan ini memvalidasi aritmetika game; efficiency saja tidak membuktikan semantik atau usefulness.

In [ ]:
phi=local_attributions[("gru","source_calendar")]["shapley"]
v=local_attributions[("gru","source_calendar")]["game"]
permutation_phi=np.zeros((2,7))
for permutation in PERMUTATIONS:
    mask=0
    for j in permutation:
        permutation_phi[:,j]+=v[mask+(1<<j)]-v[mask]
        mask+=1<<j
permutation_phi/=len(PERMUTATIONS)
assert np.allclose(phi,permutation_phi,rtol=1e-8,atol=1e-10)
print("Max independent Shapley difference:",float(abs(phi-permutation_phi).max()))

## 12. Integrated Gradients: integrasi full GB function

Gauss–Legendre 32,64,128,256,512 nodes. Accept hanya jika completeness ≤1e-8 +1e-3×|contrast| dan successive L1 change ≤1e-8 +1e-3×L1 credits. Failed attribution tetap direkam dan tidak dipakai untuk quality score. Ini numerical acceptance, bukan jaminan straight path berada pada distribusi traffic alami.

In [ ]:
NODES={n:leggauss(n) for n in [32,64,128,256,512]}

In [ ]:
def integrated_gradients(model, x, r):
    delta = x - r
    prev = None
    accepted = np.zeros(2, bool)
    answer = np.zeros((2, 168))
    steps = np.zeros(2, int)
    errors = np.zeros(2)
    lastchange = np.zeros(2)
    contrast = model(x) - model(r)
    for n, (ab, w) in NODES.items():
        alpha = (ab + 1) / 2
        ww = w / 2
        current = np.zeros((2, 168))
        for a in range(0, n, 64):
            raw = r[None, :] + alpha[a:a + 64, None] * delta[None, :]
            p = torch.tensor(raw, dtype=torch.float64, requires_grad=True)
            z = model.net(p)
            for k in range(2):
                grad = torch.autograd.grad(z[:, k].sum(), p, retain_graph=k == 0)[0].detach().numpy()
                current[k] += np.sum(ww[a:a + 64, None] * grad, axis=0) * delta
        err = abs(current.sum(1) - contrast)
        if prev is not None:
            change = abs(current - prev).sum(1)
            ok = (err <= 1e-08 + 0.001 * abs(contrast)) & (change <= 1e-08 + 0.001 * abs(current).sum(1))
            new = ok & ~accepted
            answer[new] = current[new]
            steps[new] = n
            errors[new] = err[new]
            lastchange[new] = change[new]
            accepted |= ok
        prev = current
        if accepted.all():
            break
    remaining = ~accepted
    answer[remaining] = current[remaining]
    steps[remaining] = n
    errors[remaining] = err[remaining]
    lastchange[remaining] = change[remaining]
    return (answer.reshape(2, 7, 24).sum(2), accepted, steps, errors, lastchange)

In [ ]:
local_ig={}; ig_case_rows=[]
for kind,model in predictors.items():
    for ref,r in references.items():
        credits,accepted,nodes,errors,changes=integrated_gradients(model,x_case,r)
        local_ig[(kind,ref)]={"credits":credits,"accepted":accepted}
        for k,h in enumerate(HORIZONS):ig_case_rows.append(dict(model=kind,reference=ref,horizon=h,
            accepted=bool(accepted[k]),nodes=int(nodes[k]),completeness_error_gb=float(errors[k]),l1_refinement_gb=float(changes[k])))
local_ig_audit=pd.DataFrame(ig_case_rows)
display(local_ig_audit)

In [ ]:
# Compare signed local credits only on accepted IG cases; no forced agreement assertion.
fig,axes=plt.subplots(1,3,figsize=(14,3.6),constrained_layout=True)
for ax,kind in zip(axes,["ridge","dlinear","gru"]):
    a=local_attributions[(kind,"source_calendar")]["shapley"][0]
    igcase=local_ig[(kind,"source_calendar")]
    positions=np.arange(7)
    ax.bar(positions-.18,a,width=.36,label="Shapley")
    if igcase["accepted"][0]:
        ax.bar(positions+.18,igcase["credits"][0],width=.36,label="Accepted IG")
    else:
        ax.text(.05,.93,"IG rejected numerically",transform=ax.transAxes,color="#c16b24",va="top")
    ax.set_xticks(positions,DAY_NAMES,rotation=45);ax.axhline(0,c="gray",lw=.7)
    ax.set_ylabel("Signed daily credit, GB");ax.set_title(kind);ax.legend(fontsize=8)
plt.show()

## 13. Controlled replacement operators — berbeda dari attribution references

Interpolation mengganti internal shape suatu hari dengan garis antara first/last value; endpoint information tetap ada. Rotation memakai cyclic shift satu hari dari observed week. Keduanya tanpa future labels tetapi dapat menghasilkan history yang tidak alami. Kita tidak menyebutnya causal interventions atau conditional-distribution samples.

In [ ]:
def templates(x):
    b = x.reshape(7, 24)
    return {'interpolation': np.stack([np.linspace(a[0], a[-1], 24) for a in b]).ravel(), 'rotation': np.roll(b, 1, axis=0).ravel()}

def response_game(model, x, r, bias=None):
    a = x[None, :] + np.repeat(MASKS, 24, axis=1) * (r - x)[None, :]
    v = model(a, bias)
    response = abs(v[REMOVED[:, 1:]] - v[REMOVED[:, :-1]])
    cum = np.cumsum(response, axis=1)
    utilities = cum[:, [0, 1, 3], :].mean(1)
    return (v, utilities.mean(0), utilities.max(0))

def quality(v, rank, random, oracle):
    masks = np.r_[0, np.cumsum(1 << rank)]
    cpd = np.cumsum(abs(v[masks[1:]] - v[masks[:-1]]), axis=0)
    u = cpd[[0, 1, 3]].mean(0)
    gap = oracle - random
    q = np.divide(u - random, gap, out=np.full(2, np.nan), where=gap > 1e-09)
    assert np.all(q[np.isfinite(q)] <= 1.00001), q
    return (u, q, gap)

In [ ]:
operator_templates=templates(x_case)
fig,axes=plt.subplots(2,1,figsize=(12,5),sharex=True,constrained_layout=True)
for ax,(name,t) in zip(axes,operator_templates.items()):
    ax.plot(x_case,c="black",lw=1,label="Original")
    ax.plot(t,c="#c16b24",lw=1,label=name)
    for boundary in range(24,168,24):ax.axvline(boundary,c="gray",alpha=.2)
    shift=np.sqrt(np.mean((np.log1p(t)-np.log1p(x_case))**2))
    ax.set_title(f"{name}; log-input RMS shift={shift:.4f}");ax.set_ylabel("GB");ax.legend()
axes[-1].set_xlabel("History hour");plt.show()

## 14. Menghitung Q langsung untuk satu kasus

Untuk ranking π, cumulative prediction difference Ck menjumlahkan **absolute sequential** changes ketika first k groups diganti. Utility U = mean(C1,C2,C4). Dari semua 5040 orderings, R adalah exact mean utility dan O adalah maximum response utility. Q=(U−R)/(O−R), undefined jika gap ≤1e-9 GB. Negative Q boleh; O bukan oracle forecast accuracy.

In [ ]:
local_quality_rows=[]; response_cache={}
model=predictors["gru"]
phi=local_attributions[("gru","source_calendar")]["shapley"]
occ=local_attributions[("gru","source_calendar")]["occlusion"]
igcase=local_ig[("gru","source_calendar")]
for op,t in operator_templates.items():
    values,random_mean,oracle=response_game(model,x_case,t)
    response_cache[op]=(values,random_mean,oracle)
    for k,h in enumerate(HORIZONS):
        key=f"{case_id}:{case.origin}:{h}"
        rankings={"shapley":order(abs(phi[k]),key),"occlusion":order(abs(occ[k]),key),"recency":np.arange(6,-1,-1)}
        if igcase["accepted"][k]:rankings["ig"]=order(abs(igcase["credits"][k]),key)
        for method,rank in rankings.items():
            utility,q,gap=quality(values,rank,random_mean,oracle)
            local_quality_rows.append(dict(operator=op,method=method,horizon=h,utility_gb=float(utility[k]),
                random_gb=float(random_mean[k]),oracle_gb=float(oracle[k]),gap_gb=float(gap[k]),quality_q=float(q[k])))
local_quality=pd.DataFrame(local_quality_rows)
display(local_quality)

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
for ax,(op,(values,random_mean,oracle)) in zip(axes,response_cache.items()):
    sequential=abs(values[REMOVED[:,1:],0]-values[REMOVED[:,:-1],0])
    utilities=np.cumsum(sequential,axis=1)[:,[0,1,3]].mean(1)
    ax.hist(utilities,bins=30,color="#94a3b8",label="All 5040 orderings")
    ax.axvline(random_mean[0],c="#2365a0",label="Random mean")
    ax.axvline(oracle[0],c="#c16b24",label="Response oracle")
    ax.set_title(f"{op}, GRU h1");ax.set_xlabel("Utility GB");ax.legend(fontsize=8)
plt.show()

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
for ax,(op,(values,random_mean,oracle)) in zip(axes,response_cache.items()):
    for method,rank in {"shapley":order(abs(phi[0]),f"{case_id}:{case.origin}:1"),
                        "occlusion":order(abs(occ[0]),f"{case_id}:{case.origin}:1"),
                        "recency":np.arange(6,-1,-1)}.items():
        masks=np.r_[0,np.cumsum(1<<rank)]
        cpd=np.r_[0,np.cumsum(abs(values[masks[1:],0]-values[masks[:-1],0]))]
        ax.plot(range(8),cpd,marker="o",label=method)
    ax.set_xlabel("Number of sequentially replaced day groups");ax.set_ylabel("Cumulative absolute response GB")
    ax.set_title(op);ax.legend()
plt.show()

### 14.1 Cocokkan recomputation dengan hasil tersimpan

Pilih kasus yang sama, model/seed/horizon/reference/operator sama, dan shared tie key. Tidak cukup membandingkan average berbeda subjek. Kasus undefined tetap undefined.

In [ ]:
saved_case_quality=pd.read_csv(ROOT/"results/quality_gru_42.csv.gz")
z=saved_case_quality[(saved_case_quality.id==case_id)&(saved_case_quality.origin==case.origin)&
    (saved_case_quality.arm=="frozen")&(saved_case_quality.reference=="source_calendar")]
checks=[]
for row in local_quality.itertuples():
    saved=z[(z.operator==row.operator)&(z.method==row.method)&(z.horizon==row.horizon)].iloc[0]
    ok=np.isclose(row.quality_q,saved.quality_q,rtol=1e-8,atol=1e-10,equal_nan=True)
    checks.append(dict(method=row.method,horizon=row.horizon,operator=row.operator,match=bool(ok)))
assert all(r["match"] for r in checks)
display(pd.DataFrame(checks))

## 15. Global attribution: pola per grup, cohort dan reference

Global importance dihitung dari mean absolute group credit, lalu equal institution-year means. Ini descriptive importance **pada sampled eligible origins**, bukan population causal effect. Tidak mencampur signed Shapley credits dengan occlusion magnitudes sebagai besaran yang identik.

In [ ]:
allocation_files=sorted((ROOT/"results").glob("allocations_*.csv.gz"))
allocations=pd.concat([pd.read_csv(p) for p in allocation_files],ignore_index=True)
score_cols=[f"score_{i}" for i in range(7)]
alloc=allocations.query("model=='gru' and arm=='frozen' and horizon==1 and method=='shapley'").copy()
alloc[score_cols]=alloc[score_cols].abs()
institution_importance=alloc.groupby(["id","cohort","year","reference"])[score_cols].mean().reset_index()
global_importance=institution_importance.groupby(["cohort","reference"])[score_cols].mean()
display(global_importance)
fig,axes=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
for ax,ref in zip(axes,["source_calendar","history_hour_median"]):
    pos=np.arange(7)
    for cohort,dx,col in [("source",-.18,"#2365a0"),("target",.18,"#c16b24")]:
        values=global_importance.loc[(cohort,ref)].to_numpy()
        ax.bar(pos+dx,values,width=.36,label=cohort,color=col)
    ax.set_xticks(pos,DAY_NAMES,rotation=45);ax.set_ylabel("Mean absolute group credit GB")
    ax.set_title(ref);ax.legend()
plt.show()

## 16. Agreement metode dan kestabilan antar-training-seed

Top-3 Jaccard dan Kendall τ-b berikut membandingkan rankings untuk case/reference/horizon yang sama. High agreement tidak membuktikan faithfulness; methods bisa sepakat karena alasan yang salah. Antar-training-seed mengukur variasi model training, bukan stochastic rerun instability LIME (LIME tidak digunakan di studi ini).

In [ ]:
def ranking_agreement(a,b,key,top_k=3):
    ra=order(abs(np.asarray(a)),key);rb=order(abs(np.asarray(b)),key)
    ta=set(ra[:top_k]);tb=set(rb[:top_k])
    jaccard=len(ta&tb)/len(ta|tb)
    # Average ties in magnitude are kept for tau-b; deterministic tie keys only serve top-k.
    tau=float(kendalltau(abs(np.asarray(a)),abs(np.asarray(b))).statistic)
    return jaccard,tau

agreement_rows=[]
sub=allocations.query("model=='gru' and arm=='frozen' and horizon==1 and reference=='source_calendar'")
for key,g in sub.groupby(["id","seed","year","origin","cohort"]):
    bymethod={r.method:np.array([getattr(r,col) for col in score_cols]) for r in g.itertuples()}
    if "shapley" not in bymethod or "occlusion" not in bymethod:continue
    tie=f"{key[0]}:{key[3]}:1"
    j,t=ranking_agreement(bymethod["shapley"],bymethod["occlusion"],tie)
    agreement_rows.append(dict(id=key[0],seed=key[1],year=key[2],cohort=key[4],jaccard_top3=j,kendall_tau=t))
agreement_df=pd.DataFrame(agreement_rows)
# Average within institution-year before summary; counts are descriptive, not inferential N.
agreement_macro=agreement_df.groupby(["id","year","cohort"])[["jaccard_top3","kendall_tau"]].mean()
display(agreement_macro.groupby("cohort").mean())

In [ ]:
seed_agreement_rows=[]
for key,g in sub.query("method=='shapley'").groupby(["id","year","origin","cohort"]):
    byseed={int(r.seed):np.array([getattr(r,col) for col in score_cols]) for r in g.itertuples()}
    if set(byseed)!={42,123,2026}:continue
    for a,b in [(42,123),(42,2026),(123,2026)]:
        j,t=ranking_agreement(byseed[a],byseed[b],f"{key[0]}:{key[2]}:1")
        seed_agreement_rows.append(dict(id=key[0],year=key[1],cohort=key[3],seed_pair=f"{a}/{b}",jaccard_top3=j,kendall_tau=t))
seed_agreement_df=pd.DataFrame(seed_agreement_rows)
seed_macro=seed_agreement_df.groupby(["id","year","cohort","seed_pair"])[["jaccard_top3","kendall_tau"]].mean().reset_index()
display(seed_macro.groupby(["cohort","seed_pair"])[["jaccard_top3","kendall_tau"]].mean())
fig,ax=plt.subplots(figsize=(8,4),constrained_layout=True)
cohorts=["source","target"]
ax.boxplot([seed_macro.query("cohort==@cohort").jaccard_top3 for cohort in cohorts],tick_labels=cohorts)
ax.set_ylabel("Mean top-3 Jaccard across seed-pair rankings")
ax.set_title("Model-seed stability: descriptive institution-year summaries")
plt.show()

## 17. Full numerical acceptance dan common-case comparisons

Jangan mengadu all-case Shapley dengan IG hanya pada subset yang lolos. Untuk plot komparatif berikut, pasangan metode memakai **intersection cases dengan Q defined untuk kedua metode**. Common-case plot bersifat eksploratori dan tidak mengganti primary all-case Shapley analysis.

In [ ]:
quality_paths=sorted((ROOT/"results").glob("quality_*.csv.gz"))
quality_all=pd.concat([pd.read_csv(p) for p in quality_paths],ignore_index=True)
numerics=pd.concat([pd.read_csv(p).assign(model=p.stem.split("_")[2],seed=int(p.stem.split("_")[-1]))
                   for p in sorted((ROOT/"results").glob("ig_numerics_*.csv"))],ignore_index=True)
ig_totals=numerics.groupby("model").agg(cases=("accepted","size"),accepted=("accepted","sum"))
ig_totals["failed"]=ig_totals.cases-ig_totals.accepted
ig_totals["acceptance_rate"]=ig_totals.accepted/ig_totals.cases
display(ig_totals)
assert (~numerics.accepted).sum()==claims["ig_failed"]
assert quality_all.query("method=='ig' and not accepted").quality_q.isna().all()
fig,ax=plt.subplots(figsize=(7,3.5),constrained_layout=True)
ax.bar(ig_totals.index,ig_totals.acceptance_rate,color="#0d9488")
ax.set_ylim(0,1.05);ax.set_ylabel("Numerical acceptance fraction")
ax.set_title("Fixed completeness + L1-refinement criteria")
plt.show()

In [ ]:
compare=quality_all.query("model=='gru' and arm=='frozen' and horizon==1 and reference=='source_calendar' and method in ['shapley','ig']")
common_keys=["id","seed","year","origin","cohort","operator"]
common=compare.pivot_table(index=common_keys,columns="method",values="quality_q").dropna(subset=["shapley","ig"]).reset_index()
common_seed_keys=["id","year","origin","cohort","operator"]
seed_counts=common.groupby(common_seed_keys)["seed"].transform("nunique")
common=common[seed_counts==3].copy()
common_seed_average=common.groupby(common_seed_keys)[["shapley","ig"]].mean().reset_index()
common_macro=common_seed_average.groupby(["id","year","cohort"])[["shapley","ig"]].mean().reset_index()
print("Matched operator/seed cases (complete 3-seed sets):",len(common),"; available institution-year cells:",len(common_macro))
display(common_macro.groupby("cohort")[["shapley","ig"]].mean())
fig,ax=plt.subplots(figsize=(6,4),constrained_layout=True)
for cohort,color in [("source","#2365a0"),("target","#c16b24")]:
    z=common_macro.query("cohort==@cohort")
    ax.scatter(z.shapley,z.ig,color=color,label=cohort)
lo=min(common_macro.shapley.min(),common_macro.ig.min());hi=max(common_macro.shapley.max(),common_macro.ig.max())
ax.plot([lo,hi],[lo,hi],c="gray",ls="--");ax.set_xlabel("Shapley mean Q on common cases")
ax.set_ylabel("IG mean Q on common cases");ax.legend();plt.show()

## 18. Primary paired estimate: seed completeness dan exact aggregation

Semua tiga neural seeds harus memiliki defined Q; Ridge cukup satu. Pasangan source/target harus punya timestamp yang sama. Empat cells (dua tahun × dua operators) harus tersedia untuk pair-level primary contribution. Data yang tidak lengkap disimpan, tidak diganti pasangan yang memberi hasil lebih menarik.

In [ ]:
seed_keys=["model","id","pair","cohort","year","arm","horizon","origin","reference","method","operator","ig_subset"]
seed_average=quality_all.groupby(seed_keys,dropna=False).quality_q.agg(["mean","count"]).reset_index()
seed_average["expected"]=np.where(seed_average.model.eq("ridge"),1,3)
seed_average.loc[seed_average["count"]!=seed_average.expected,"mean"]=np.nan
seed_average=seed_average.rename(columns={"mean":"quality_q"})
pair_keys=["model","pair","year","arm","horizon","origin","reference","method","operator","ig_subset"]
paired_quality=seed_average.pivot_table(index=pair_keys,columns="cohort",values="quality_q").reset_index()
paired_quality=paired_quality.dropna(subset=["source","target"])
paired_quality["delta"]=paired_quality.target-paired_quality.source
primary=paired_quality.query("model=='gru' and arm=='frozen' and horizon==1 and reference=='source_calendar' and method=='shapley'").copy()
cell_means=primary.groupby(["pair","year","operator"])[["source","target","delta"]].mean()
complete_pairs=cell_means.delta.unstack(["year","operator"]).dropna().index
primary_complete=primary[primary["pair"].isin(complete_pairs)]
point=cell_means.loc[complete_pairs].groupby("pair").mean().mean()
print("Paired origin/operator records before complete-pair restriction:",len(primary))
print("Records after complete-pair restriction:",len(primary_complete))
print("Complete pair IDs:",complete_pairs.tolist())
display(point.rename("equal-cell/equal-pair mean").to_frame())
assert np.isclose(point.delta,claims["primary"]["mean"],atol=1e-12)

### 18.1 Paired institution/14-day-block bootstrap — kode dan penghitungan ulang

Resample pairs, lalu observed 14-day blocks dalam pair/year. Source/target/operator dibawa bersama. Draw dengan operator cell kosong dianggap invalid. Interval hanya jika ≥4 complete pairs dan ≥1000 valid dari 2000 attempts. Seeds dan banyak model queries bukan penambahan independent sample size. Seven related pairs membatasi kepastian interval.

In [ ]:
def bootstrap(frame, B=2000):
    f = frame.copy()
    f['block'] = (pd.to_datetime(f.origin, utc=True) - pd.to_datetime(f.year.astype(str) + '-04-08', utc=True)).dt.days // 14
    cells = f.groupby(['pair', 'year', 'operator']).size()
    good = [p for p in f.pair.unique() if len(cells.loc[p]) == 4]
    f = f[f.pair.isin(good)]
    point = float(f.groupby(['pair', 'year', 'operator']).delta.mean().groupby('pair').mean().mean()) if len(f) else None
    if len(good) < 4:
        return dict(mean=point, ci_low=None, ci_high=None, pairs=len(good), bootstrap_valid=0, reason='fewer_than_four_complete_pairs')
    arrays = {}
    for p in good:
        arrays[p] = {}
        for year in [2025, 2026]:
            g = f[(f.pair == p) & (f.year == year)]
            blocks = sorted(g.block.unique())
            a = g.groupby(['block', 'operator']).delta.agg(['sum', 'count']).unstack('operator').reindex(blocks).fillna(0)
            arrays[p][year] = (a['sum'].reindex(columns=['interpolation', 'rotation']).to_numpy(), a['count'].reindex(columns=['interpolation', 'rotation']).to_numpy())
    rng = np.random.default_rng(20261105)
    vals = []
    for _ in range(B):
        pair_stats = []
        valid = True
        for p in rng.choice(good, len(good), replace=True):
            means = []
            for year in [2025, 2026]:
                sums, counts = arrays[p][year]
                ix = rng.integers(len(sums), size=len(sums))
                den = counts[ix].sum(0)
                if (den == 0).any():
                    valid = False
                    break
                means.extend((sums[ix].sum(0) / den).tolist())
            if not valid:
                break
            pair_stats.append(np.mean(means))
        if valid:
            vals.append(np.mean(pair_stats))
    return dict(mean=point, ci_low=float(np.quantile(vals, 0.025)) if len(vals) >= 1000 else None, ci_high=float(np.quantile(vals, 0.975)) if len(vals) >= 1000 else None, pairs=len(good), bootstrap_valid=len(vals), reason='nominal_paired_pair_and_14day_block_bootstrap')

In [ ]:
bootstrap_result=bootstrap(primary,B=2000)
print(json.dumps(bootstrap_result,indent=2))
for key in ["mean","ci_low","ci_high"]:
    assert np.isclose(bootstrap_result[key],claims["primary"][key],atol=1e-12)
assert bootstrap_result["bootstrap_valid"]==claims["primary"]["bootstrap_valid"]
print("Frozen primary bootstrap reproduced.")

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
z=cell_means.loc[complete_pairs].groupby(["pair","operator"]).mean().reset_index()
for op,dx,marker in [("interpolation",-.08,"o"),("rotation",.08,"s")]:
    q=z.query("operator==@op")
    axes[0].scatter(q["pair"]+dx,q.delta,marker=marker,label=op)
axes[0].axhline(0,c="gray");axes[0].set_xticks(complete_pairs)
axes[0].set_xlabel("Complete institution pair");axes[0].set_ylabel("Target − source Q");axes[0].legend()
mean=bootstrap_result["mean"];low=bootstrap_result["ci_low"];high=bootstrap_result["ci_high"]
axes[1].errorbar([mean],[0],xerr=[[mean-low],[high-mean]],fmt="o",capsize=7)
axes[1].axvline(0,c="gray",ls="--");axes[1].set_yticks([0],["Primary GRU h1"])
axes[1].set_xlabel("Target − source Q; nominal 95% interval")
plt.show()

## 19. Secondary reference/horizon sensitivity

Plot di bawah menunjukkan nominal exploratory intervals, tanpa multiplicity correction. Hasil history-median yang lebih negatif tidak dipromosikan sebagai primary setelah melihat outcome. Reference berbeda juga berarti explanatory contrast berbeda.

In [ ]:
secondary=pd.read_csv(ROOT/"results/secondary_quality_contrasts.csv")
display(secondary.round(6))
fig,axes=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
for ax,ref in zip(axes,["source_calendar","history_hour_median"]):
    z=secondary.query("reference==@ref").sort_values(["model","horizon"]).reset_index(drop=True)
    y=np.arange(len(z))
    ax.errorbar(z["mean"],y,xerr=np.vstack([z["mean"]-z.ci_low,z.ci_high-z["mean"]]),fmt="o",capsize=3)
    ax.set_yticks(y,[f"{r.model} {r.horizon}h" for r in z.itertuples()])
    ax.axvline(0,c="gray",ls="--");ax.set_title(ref);ax.set_xlabel("Exploratory target − source Q")
plt.show()

## 20. Bias calibration dan robustness

Calibration menggunakan data lokal; arm ini bukan zero-shot. Perubahan MASE dan Q bukan kuantitas yang sama. Tidak menyimpulkan kausalitas antara membaiknya forecast dan membaiknya explanation dari dua descriptive averages.

In [ ]:
calibration_offsets=pd.read_csv(ROOT/"results/calibration_offsets.csv")
calibrated_table=macro.query("role=='target' and model in ['ridge','dlinear','gru']").pivot_table(index=["model","horizon"],columns="arm",values="mase")
calibrated_table["calibrated_minus_frozen"]=calibrated_table.calibrated-calibrated_table.frozen
display(calibrated_table.round(6))
display(calibration_offsets.groupby(["model","status"]).size().rename("offset_records").to_frame())
cal_q=pd.read_csv(ROOT/"results/calibration_quality_changes.csv.gz")
cal_q_summary=cal_q.query("cohort=='target' and horizon==1 and reference=='source_calendar'").groupby(["model","id","year"]).delta.mean().groupby("model").mean()
display(cal_q_summary.rename("Exploratory target calibration ΔQ").to_frame())

## 21. Explanation latency — benchmark lokal tambahan

Bandingkan satu history/reference yang sama, dua endpoint sekaligus, CPU empat threads. Shapley menghitung 128 coalitions; occlusion hanya membutuhkan full + tujuh single-group replacement; IG memakai adaptive node budget yang sama. Angka ini microbenchmark deskriptif, bukan hasil general latency study atau angka yang diam-diam ditambahkan ke paper.

In [ ]:
def occlusion_only(model,x,r):
    inputs=np.repeat(x[None,:],8,axis=0)
    for j in range(7):inputs[j+1,j*24:(j+1)*24]=r[j*24:(j+1)*24]
    pred=model(inputs)
    return abs(pred[0]-pred[1:]).T

latency_rows=[]
latency_model=predictors["gru"];latency_ref=references["source_calendar"]
assert np.allclose(occlusion_only(latency_model,x_case,latency_ref),local_attributions[("gru","source_calendar")]["occlusion"])
for method,fn in [("Exact 7-group Shapley",lambda:games(latency_model,x_case,latency_ref)),
                  ("Single-group occlusion",lambda:occlusion_only(latency_model,x_case,latency_ref)),
                  ("Adaptive IG",lambda:integrated_gradients(latency_model,x_case,latency_ref))]:
    fn()  # warm-up excluded
    for repeat in range(3):
        t0=time.perf_counter();fn();elapsed=time.perf_counter()-t0
        latency_rows.append(dict(method=method,repeat=repeat,seconds=elapsed))
latency_df=pd.DataFrame(latency_rows)
display(latency_df.groupby("method").seconds.agg(["median","min","max"]))
fig,ax=plt.subplots(figsize=(9,3),constrained_layout=True)
z=latency_df.groupby("method").seconds.median()
ax.barh(z.index,z.values,color="#2365a0");ax.set_xlabel("Median seconds / case (both endpoints)")
plt.show()

## 22. Full pipeline evaluation — implementasi inline untuk rerun

Tiga fungsi di bawah adalah mirror kode batch canonical: forecast metrics, attribution evaluation, dan statistical analysis. Mereka tidak dieksekusi pada mode default. Kode tampak penuh dan menggunakan fungsi yang sudah didefinisikan di notebook, bukan memanggil subprocess script. Bagian berikut mengarahkan penulisan ke clone baru.

Fungsi analysis menyimpan beberapa historical descriptive figures; untuk primary panel yang benar gunakan aggregation complete pairs pada Bagian 18. Local added analyses tetap exploratory.

In [ ]:
def evaluate_forecasts_inline():
    start = time.time()
    (ROOT / 'predictions').mkdir(exist_ok=True)
    split = pd.read_csv(ROOT / 'INSTITUTION_SPLIT.csv')
    split = split[split.role != 'validation']
    cache = {int(r.id): windows(ROOT / f'data/institution_{r.id}.csv.gz') for r in split.itertuples()}
    metrics = []
    cal = []
    specs = [(k, 0) for k in ['persistence', 'daily', 'weekly', 'ridge']] + [(k, s) for k in ['dlinear', 'gru'] for s in [42, 123, 2026]]
    for kind, seed in specs:
        model = Predictor(kind, seed) if kind in ['ridge', 'dlinear', 'gru'] else None
        frames = []
        for row in split.itertuples():
            v = cache[row.id]
            den, floor = denominator(v)
            for year in YEARS:
                q = period(v, f'{year}-04-08', f'{year}-07-01', 6)
                n = int(q.sum())
                if not n:
                    continue
                bias, ncal = calibration(v, year, model, kind)
                base = model(v['X'][q], rawlog=True) if model is not None else np.log1p(naive(v, q, kind))
                truth = v['Y'][q]
                dates = v['t'][v['o'][q]].astype(str)
                for k, h in enumerate([1, 6]):
                    cal.append(dict(model=kind, seed=seed, id=row.id, year=year, horizon=h, n_calibration=ncal, bias_log=float(bias[k]), status='ok' if ncal >= 24 else 'insufficient'))
                for arm, b in [('frozen', np.zeros(2)), ('calibrated', bias)]:
                    pred = np.expm1(np.maximum(base + b, 0))
                    assert np.isfinite(pred).all() and (pred >= 0).all()
                    for k, h in enumerate([1, 6]):
                        err = pred[:, k] - truth[:, k]
                        metrics.append(dict(model=kind, seed=seed, id=row.id, role=row.role, year=year, arm=arm, horizon=h, n=n, mae_gb=float(abs(err).mean()), rmse_gb=float(np.sqrt(np.mean(err ** 2))), mase=float(abs(err).mean() / den), mase_denominator_gb=den, denominator_floored=floor, negative_fraction=float((pred[:, k] < 0).mean()), zero_fraction=float((pred[:, k] == 0).mean())))
                        frames.append(pd.DataFrame(dict(model=kind, seed=seed, id=row.id, role=row.role, year=year, arm=arm, horizon=h, origin=dates, target_gb=truth[:, k], prediction_gb=pred[:, k])))
        pd.concat(frames, ignore_index=True).to_csv(ROOT / f'predictions/{kind}_{seed}.csv.gz', index=False, compression={'method': 'gzip', 'mtime': 0})
        pd.DataFrame(metrics).to_csv(ROOT / 'results/forecast_metrics.csv', index=False)
        pd.DataFrame(cal).to_csv(ROOT / 'results/calibration_offsets.csv', index=False)
        print('Forecasts', kind, seed, 'complete', flush=True)
    facts = dict(status='complete', runtime_seconds=time.time() - start, specs=specs, metric_rows=len(metrics), predictions_files=len(specs), target_outcomes_scored=True, evaluation_periods=YEARS)
    (ROOT / 'results/FORECAST_EVALUATION.json').write_text(json.dumps(facts, indent=2))
    print(json.dumps(facts, indent=2))

In [ ]:
def evaluate_attributions_inline():
    start = time.time()
    pairs = pd.read_csv(ROOT / 'MATCHED_PAIRS.csv')
    ids = set(pairs.source) | set(pairs.target)
    cache = {int(i): windows(ROOT / f'data/institution_{i}.csv.gz') for i in ids}
    profile = np.load(ROOT / 'models/source_profile.npy')
    tasks = []
    selection = []
    for pair in pairs.itertuples():
        for year in [2025, 2026]:
            members = {}
            for ident in [pair.source, pair.target]:
                v = cache[ident]
                q = period(v, f'{year}-04-08', f'{year}-07-01', 6)
                ix = np.where(q)[0]
                members[ident] = {int(v['o'][i]): int(i) for i in ix}
            common = sorted(set(members[pair.source]) & set(members[pair.target]))
            take = np.unique(np.linspace(0, len(common) - 1, min(16, len(common)), dtype=int)) if common else np.array([], int)
            chosen = [common[i] for i in take]
            ig_ix = set(np.unique(np.linspace(0, len(chosen) - 1, min(4, len(chosen)), dtype=int))) if chosen else set()
            for j, o in enumerate(chosen):
                for ident, cohort in [(pair.source, 'source'), (pair.target, 'target')]:
                    tasks.append(dict(pair=int(pair.pair), id=int(ident), cohort=cohort, year=year, o=int(o), window_ix=members[ident][o], ig=j in ig_ix))
                    selection.append(dict(pair=int(pair.pair), id=int(ident), cohort=cohort, year=year, origin=str(cache[ident]['t'][o]), ig=j in ig_ix, common_eligible_origins=len(common)))
    pd.DataFrame(selection).to_csv(ROOT / 'results/explanation_selection.csv', index=False)
    specs = [('ridge', 0)] + [(k, s) for k in ['dlinear', 'gru'] for s in [42, 123, 2026]]
    all_summary = []
    for kind, seed in specs:
        model = Predictor(kind, seed)
        rows = []
        allocations = []
        numeric = []
        offsets = {(i, y): calibration(cache[i], y, model)[0] for i in ids for y in [2025, 2026]}
        for n, task in enumerate(tasks):
            ident = task['id']
            v = cache[ident]
            o = task['o']
            x = v['X'][task['window_ix']]
            origin = str(v['t'][o])
            how = v['t'].day_of_week.to_numpy() * 24 + v['t'].hour.to_numpy()
            references = {'source_calendar': profile[how[o - 167:o + 1]], 'history_hour_median': np.tile(np.median(x.reshape(7, 24), axis=0), 7)}
            key = f'{ident}:{origin}'
            info = {k: task[k] for k in ['pair', 'id', 'cohort', 'year']} | dict(model=kind, seed=seed, origin=origin, ig_subset=task['ig'])
            ig_results = {name: integrated_gradients(model, x, r) for name, r in references.items()} if task['ig'] else {}
            for arm, bias in [('frozen', np.zeros(2)), ('calibrated', offsets[ident, task['year']])]:
                responses = {}
                for op, r in templates(x).items():
                    vv, rr, oracle = response_game(model, x, r, bias)
                    shift = float(np.sqrt(np.mean((np.log1p(r) - np.log1p(x)) ** 2)))
                    responses[op] = (vv, rr, oracle, shift)
                for reference, r in references.items():
                    phi, occ, game, eff = games(model, x, r, bias)
                    methods = {'shapley': (phi, np.ones(2, bool)), 'occlusion': (occ, np.ones(2, bool)), 'recency': (np.repeat(np.arange(7)[None, :], 2, axis=0), np.ones(2, bool))}
                    if arm == 'frozen' and task['ig']:
                        val, ok, nodes, err, change = ig_results[reference]
                        methods['ig'] = (val, ok)
                        for k, h in enumerate([1, 6]):
                            numeric.append(info | dict(reference=reference, horizon=h, accepted=bool(ok[k]), nodes=int(nodes[k]), completeness_error_gb=float(err[k]), l1_refinement_change_gb=float(change[k])))
                    for method, (scores, accepted) in methods.items():
                        for k, h in enumerate([1, 6]):
                            rank = np.arange(6, -1, -1) if method == 'recency' else order(abs(scores[k]), f'{key}:{h}')
                            base = info | dict(arm=arm, reference=reference, method=method, horizon=h, accepted=bool(accepted[k]))
                            allocations.append(base | {f'score_{j}': float(scores[k, j]) for j in range(7)} | dict(shapley_efficiency_error_gb=float(eff[k]) if method == 'shapley' else np.nan))
                            for op, (vv, rr, oracle, shift) in responses.items():
                                u, q, gap = quality(vv, rank, rr, oracle)
                                rows.append(base | dict(operator=op, utility_gb=float(u[k]), random_utility_gb=float(rr[k]), oracle_utility_gb=float(oracle[k]), oracle_gap_gb=float(gap[k]), quality_q=float(q[k]) if accepted[k] else np.nan, discriminating=bool(gap[k] > 1e-09), template_log_rms_shift=shift))
            if (n + 1) % 64 == 0:
                print(kind, seed, 'cases', n + 1, '/', len(tasks), 'elapsed', round(time.time() - start, 1), flush=True)
        stem = f'{kind}_{seed}'
        pd.DataFrame(rows).to_csv(ROOT / f'results/quality_{stem}.csv.gz', index=False, compression={'method': 'gzip', 'mtime': 0})
        pd.DataFrame(allocations).to_csv(ROOT / f'results/allocations_{stem}.csv.gz', index=False, compression={'method': 'gzip', 'mtime': 0})
        pd.DataFrame(numeric).to_csv(ROOT / f'results/ig_numerics_{stem}.csv', index=False)
        item = dict(model=kind, seed=seed, case_origins=len(tasks), quality_records=len(rows), ig_cases=len(numeric), ig_failed=sum((not r['accepted'] for r in numeric)))
        all_summary.append(item)
        print('COMPLETED', item, flush=True)
        (ROOT / 'results/ATTRIBUTION_PROGRESS.json').write_text(json.dumps(all_summary, indent=2))
    facts = dict(status='complete', runtime_seconds=time.time() - start, models=all_summary, target_outcomes_scored=True, source_sha256=hashlib.sha256(__import__('pathlib').Path(NOTEBOOK_PATH).read_bytes()).hexdigest())
    (ROOT / 'results/ATTRIBUTION_COMPLETE.json').write_text(json.dumps(facts, indent=2))
    print(json.dumps(facts, indent=2))

In [ ]:
def analyze_results_inline():
    (ROOT / 'figures').mkdir(exist_ok=True)
    m = pd.read_csv(ROOT / 'results/forecast_metrics.csv')
    q = pd.concat([pd.read_csv(p) for p in sorted((ROOT / 'results').glob('quality_*.csv.gz'))], ignore_index=True)
    num = pd.concat([pd.read_csv(p).assign(model=p.stem.split('_')[2], seed=int(p.stem.split('_')[-1])) for p in sorted((ROOT / 'results').glob('ig_numerics_*.csv'))], ignore_index=True)
    keys = ['model', 'id', 'pair', 'cohort', 'year', 'arm', 'horizon', 'origin', 'reference', 'method', 'operator', 'ig_subset']
    a = q.groupby(keys, dropna=False).quality_q.agg(['mean', 'count']).reset_index()
    a['expected'] = np.where(a.model == 'ridge', 1, 3)
    a.loc[a['count'] != a.expected, 'mean'] = np.nan
    a = a.rename(columns={'mean': 'quality_q'})
    pk = ['model', 'pair', 'year', 'arm', 'horizon', 'origin', 'reference', 'method', 'operator', 'ig_subset']
    paired = a.pivot_table(index=pk, columns='cohort', values='quality_q').reset_index()
    paired = paired.dropna(subset=['source', 'target'])
    paired['delta'] = paired.target - paired.source
    paired.to_csv(ROOT / 'results/paired_quality.csv.gz', index=False, compression={'method': 'gzip', 'mtime': 0})
    primary = paired[(paired.model == 'gru') & (paired.arm == 'frozen') & (paired.horizon == 1) & (paired.reference == 'source_calendar') & (paired.method == 'shapley')]
    claim = bootstrap(primary)
    secondary = []
    for key, g in paired[(paired.method == 'shapley') & (paired.arm == 'frozen')].groupby(['model', 'horizon', 'reference']):
        secondary.append(dict(model=key[0], horizon=int(key[1]), reference=key[2]) | bootstrap(g))
    pd.DataFrame(secondary).to_csv(ROOT / 'results/secondary_quality_contrasts.csv', index=False)
    qmacro = a.groupby(['model', 'id', 'cohort', 'year', 'arm', 'horizon', 'reference', 'method']).quality_q.mean().reset_index()
    qmacro.to_csv(ROOT / 'results/quality_institution_macro.csv', index=False)
    mseed = m.groupby(['model', 'id', 'role', 'year', 'arm', 'horizon'])[['mae_gb', 'rmse_gb', 'mase', 'negative_fraction', 'zero_fraction']].mean().reset_index()
    mseed.to_csv(ROOT / 'results/forecast_institution_macro.csv', index=False)
    mm = mseed.groupby(['model', 'role', 'arm', 'horizon'])[['mae_gb', 'rmse_gb', 'mase', 'negative_fraction', 'zero_fraction']].mean().reset_index()
    mm.to_csv(ROOT / 'results/forecast_macro.csv', index=False)
    exclusions = q.groupby(['model', 'cohort', 'year', 'arm', 'horizon', 'reference', 'method', 'operator']).agg(records=('quality_q', 'size'), defined=('quality_q', 'count'), discriminating=('discriminating', 'sum'), accepted=('accepted', 'sum')).reset_index()
    exclusions.to_csv(ROOT / 'results/quality_exclusions.csv', index=False)
    nsummary = num.groupby(['model', 'reference', 'horizon']).agg(cases=('accepted', 'size'), accepted=('accepted', 'sum'), max_completeness_error_gb=('completeness_error_gb', 'max')).reset_index()
    nsummary.to_csv(ROOT / 'results/ig_numerical_summary.csv', index=False)
    byop = primary.groupby(['pair', 'operator']).agg(source_q=('source', 'mean'), target_q=('target', 'mean'), delta=('delta', 'mean'), n=('origin', 'size')).reset_index()
    byop.to_csv(ROOT / 'results/primary_pair_operator.csv', index=False)
    ck = ['model', 'id', 'pair', 'cohort', 'year', 'horizon', 'origin', 'reference', 'method', 'operator', 'ig_subset']
    cal = a[a.method == 'shapley'].pivot_table(index=ck, columns='arm', values='quality_q').reset_index()
    cal = cal.dropna(subset=['frozen', 'calibrated'])
    cal['delta'] = cal.calibrated - cal.frozen
    cal.to_csv(ROOT / 'results/calibration_quality_changes.csv.gz', index=False, compression={'method': 'gzip', 'mtime': 0})
    names = ['persistence', 'daily', 'weekly', 'ridge', 'dlinear', 'gru']
    colors = ['#94a3b8', '#64748b', '#475569', '#2563eb', '#0d9488', '#ea580c']
    fig, axs = plt.subplots(1, 2, figsize=(10, 4), layout='constrained')
    for ax, h in zip(axs, [1, 6]):
        f = mseed[(mseed.role == 'target') & (mseed.arm == 'frozen') & (mseed.horizon == h)].groupby(['model', 'id']).mase.mean()
        means = [f.loc[k].mean() for k in names]
        ax.bar(range(6), means, color=colors)
        ax.set_xticks(range(6), ['Last', 'Daily', 'Weekly', 'Ridge', 'DLinear', 'GRU'], rotation=25)
        ax.set_ylabel('Mean institution MASE')
        ax.set_title(f'Unseen institutions: {h}h endpoint')
        ax.grid(axis='y', alpha=0.2)
    fig.savefig(ROOT / 'figures/target_forecast_mase.png', dpi=190)
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(9, 4), layout='constrained')
    ops = ['interpolation', 'rotation']
    markers = ['o', 's']
    for op, c, mark in zip(ops, ['#2563eb', '#ea580c'], markers):
        g = byop[byop.operator == op]
        ax.scatter(g.pair + (0 if op == 'interpolation' else 0.12), g.delta, label=op, color=c, marker=mark, s=45)
    ax.axhline(0, color='black', lw=1)
    ax.set_xlabel('Matched institution pair')
    ax.set_ylabel('Target - source ranking score Q')
    ax.set_title('Primary GRU / 1h / source-calendar Shapley')
    ax.legend()
    ax.grid(axis='y', alpha=0.2)
    fig.savefig(ROOT / 'figures/primary_pair_contrasts.png', dpi=190)
    plt.close(fig)
    fig, axs = plt.subplots(1, 2, figsize=(10, 4), layout='constrained')
    for ax, ref in zip(axs, ['source_calendar', 'history_hour_median']):
        z = a[(a.method == 'shapley') & (a.arm == 'frozen') & (a.horizon == 1) & (a.reference == ref)].groupby(['model', 'cohort', 'id']).quality_q.mean().reset_index()
        for j, kind in enumerate(['ridge', 'dlinear', 'gru']):
            for cohort, dx, c in [('source', -0.16, '#2563eb'), ('target', 0.16, '#ea580c')]:
                vals = z[(z.model == kind) & (z.cohort == cohort)].quality_q.to_numpy()
                ax.scatter(np.full(len(vals), j + dx), vals, color=c, s=24, alpha=0.7, label=cohort if j == 0 else None)
        ax.axhline(0, color='gray', lw=0.6)
        ax.set_xticks(range(3), ['Ridge', 'DLinear', 'GRU'])
        ax.set_ylabel('Mean Q per institution')
        ax.set_title(ref.replace('_', ' '))
        ax.grid(axis='y', alpha=0.2)
    axs[0].legend()
    fig.savefig(ROOT / 'figures/reference_quality.png', dpi=190)
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(8, 4), layout='constrained')
    labels = []
    rates = []
    for r in nsummary.itertuples():
        labels.append(f"{r.model}\n{r.reference.split('_')[0]} {r.horizon}h")
        rates.append(r.accepted / r.cases)
    ax.bar(range(len(rates)), rates, color='#0d9488')
    ax.set_xticks(range(len(rates)), labels, rotation=35, ha='right', fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('IG cases meeting numerical tolerances')
    ax.grid(axis='y', alpha=0.2)
    fig.savefig(ROOT / 'figures/ig_acceptance.png', dpi=190)
    plt.close(fig)
    claims = dict(primary=claim, primary_definition='GRU frozen, h1, source-calendar Shapley Q, target-minus-source, paired institution/time blocks', quality_records=len(q), seed_averaged_rows=len(a), paired_primary_records=len(primary), ig_cases=len(num), ig_failed=int((~num.accepted).sum()), forecast_metric_rows=len(m), target_institutions=8, matched_pairs=8, protocol='FULL_PROTOCOL.md', all_operators_controlled=True, causal_demand_claim=False)
    (ROOT / 'results/NUMERIC_CLAIMS.json').write_text(json.dumps(claims, indent=2))
    print(json.dumps(claims, indent=2))
    print(mm[(mm.role == 'target') & (mm.arm == 'frozen')].to_string(index=False))

### 22.1 Jalankan reproduksi penuh pada clone baru

Aktifkan `RUN_FULL_TRAINING` dan/atau `RUN_FULL_EVALUATION` di konfigurasi, lalu jalankan kembali sel ini. Training-only tidak otomatis menghasilkan target metrics baru. Evaluation-only menggunakan checkpoint yang disertakan. Clone menyimpan artifacts lama sebagai pembanding yang akan ditimpa per tahap; `NOTEBOOK_RERUN_CONTEXT.json` mencatat tahap yang benar-benar dijalankan agar hasil lama tidak dianggap hasil rerun.

Ini bukan eksperimen baru atau hasil confirmatory baru. Jangan mengedit frozen split/hyperparameters hanya untuk mencari target result yang lebih kuat. ROOT dikembalikan ke original setelah selesai, termasuk ketika terjadi error.

In [ ]:
if RUN_FULL_TRAINING or RUN_FULL_EVALUATION:
    stamp=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    run_root=SOURCE_ROOT.parent/"work"/"xai_notebook_reruns"/stamp
    shutil.copytree(SOURCE_ROOT,run_root,ignore=shutil.ignore_patterns("__pycache__","*.aux","*.log","*.out"))
    context={"status":"running","implementation":"inline notebook mirror","fresh_holdout":False,
             "training_requested":RUN_FULL_TRAINING,"evaluation_requested":RUN_FULL_EVALUATION,"completed_steps":[],
             "warning":"Clone contains baseline artifacts; only completed_steps are recomputed."}
    ROOT=run_root
    try:
        (ROOT/"results/NOTEBOOK_RERUN_CONTEXT.json").write_text(json.dumps(context,indent=2))
        steps=([("training",train_all_models_inline)] if RUN_FULL_TRAINING else [])
        if RUN_FULL_EVALUATION:
            steps += [("forecasts",evaluate_forecasts_inline),("attributions",evaluate_attributions_inline),("analysis",analyze_results_inline)]
        for name,fn in steps:
            print("Running inline:",name,flush=True);fn();context["completed_steps"].append(name)
            (ROOT/"results/NOTEBOOK_RERUN_CONTEXT.json").write_text(json.dumps(context,indent=2))
        context["status"]="completed"
    except Exception:
        context["status"]="failed";raise
    finally:
        (ROOT/"results/NOTEBOOK_RERUN_CONTEXT.json").write_text(json.dumps(context,indent=2))
        ROOT=SOURCE_ROOT
    print("Clone saved:",run_root)
else:
    print("Full training and batch evaluation disabled; original artifacts unchanged.")

## 23. Export analisis notebook dan konsistensi ilmiah

Export optional menulis hanya local/exploratory notebook derivatives ke folder kerja baru. Tidak mengganti main study result files. Latency, agreement, dan kasus lokal tidak dipromosikan menjadi hasil primer.

In [ ]:
if EXPORT_EXPLORATORY:
    output_dir=SOURCE_ROOT.parent/"work"/"xai_notebook_exports"/datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    output_dir.mkdir(parents=True,exist_ok=False)
    for name,frame in {"local_attribution_numerics":local_numerics,"local_ig_audit":local_ig_audit,
                       "local_quality":local_quality,"method_agreement":agreement_df,
                       "seed_agreement":seed_agreement_df,"local_latency":latency_df}.items():
        frame.to_csv(output_dir/f"{name}.csv",index=False)
    print("Exploratory exports:",output_dir)
else:
    print("Exploratory export disabled; outputs are embedded in notebook.")

# Read-only post-run consistency guards.
for item in training_record["models"]:
    assert hashlib.sha256((SOURCE_ROOT/"models"/item["file"]).read_bytes()).hexdigest()==item["sha256"]
assert json.loads((SOURCE_ROOT/"results/NUMERIC_CLAIMS.json").read_text())["primary"]==claims["primary"]
assert len(complete_pairs)==7 and len(primary)==369
assert int((~numerics.accepted).sum())==210
print("Original checkpoints and primary result preserved; notebook checks passed.")

## 24. Interpretasi, gap, dan artefak paper

**Yang didukung:** primary target−source Q sekitar −0.019 dengan nominal 95% interval yang mencakup nol; model linear kompetitif pada forecasting; reference/operator memberi hasil berbeda; finite-budget IG rejection perlu dilaporkan.

**Yang belum didukung:** degradasi universal, equivalence/non-inferiority, causal demand explanation, klaim pertama explanation transfer, superiority terhadap specialized explainers yang tidak diimplementasikan, atau generalisasi lintas ISP.

Transfer forecasting sudah dibahas [TransMUSE](https://doi.org/10.1016/j.comnet.2022.109518), temporal network explanations oleh [AIChronoLens](https://doi.org/10.1109/TMC.2025.3554035), explanation generalization oleh [TimeSAE, ICML 2026](https://arxiv.org/abs/2601.09776v2), dan temporal evaluation oleh [TIMING](https://proceedings.mlr.press/v267/jang25a.html). Kontribusi kita adalah empirical reliability audit dengan setting/control yang eksplisit, bukan algoritma baru.

- [Paper PDF](paper/paper.pdf), [LaTeX](paper/paper.tex), [PPT editable](presentation/XAI_Research_Results_v2.pptx).
- [Protocol](FULL_PROTOCOL.md), [analysis clarifications](ANALYSIS_CLARIFICATIONS.md), [literature matrix](LITERATURE_AND_GAP.md).
- [Results brief Indonesia](RESULTS_BRIEF_ID.md), [data attribution](DATA_ATTRIBUTION.md), [validation notes](VALIDATION_NOTES.md).

## 25. Pemeriksaan sebelum memakai hasil untuk submission

1. Pastikan kernel/environment, file hashes dan chronological split sesuai.
2. Pisahkan default notebook recomputations, full rerun stages, dan original paper artifacts.
3. Laporkan missing pair-year, denominator failures, failed IG dan common-case restrictions.
4. Pertahankan primary endpoint, jangan memilih reference/model signifikan setelah target dibuka.
5. Sesuaikan venue, authorship/afiliasi dan disclosure. Belum ada submission.

Notebook referensi hananlu/xai dipakai sebagai inspirasi kedalaman penyajian; task, dataset, statistical unit dan klaim ilmiah proyek ini tetap forecasting institusi.